In [1]:
# FP-GROWTH
import time
import os
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

# ---------- 1. ĐỌC DỮ LIỆU (giống hệt pipeline Apriori) ----------
XLSX = 'Mushroom_Preprocessed_Dataset_Cleaned.xlsx'   # đổi thành '/content/...' nếu chạy Colab
CSV = 'mushrooms.csv'

if os.path.exists(XLSX):
    # CHÚ Ý: phải đọc đúng sheet dữ liệu đã làm sạch (KHÔNG dùng sheet_name=0,
    # vì sheet 0 là bảng mô tả các bước tiền xử lý, chỉ có 5 dòng)
    df = pd.read_excel(XLSX, sheet_name='3. Dữ liệu đã xử lý (Cleaned)')
else:
    df = pd.read_csv(CSV)
    df = df.drop(columns=['veil-type'], errors='ignore')
    df = df.drop_duplicates()
print(f'-> Dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột')

# One-hot y hệt bước Apriori
df_encoded = pd.get_dummies(df) > 0
print(f'-> Ma trận one-hot: {df_encoded.shape}')

# ---------- 2. CẤU HÌNH: CÙNG NGƯỠNG VỚI APRIORI ----------
minsup_list = [0.3, 0.4, 0.5]
minconf_list = [0.6, 0.7, 0.8]
N_REPEAT = 3          # lặp mỗi cấu hình, lấy median để giảm nhiễu thời gian

# Warm-up (loại bỏ chi phí khởi tạo lần đầu khỏi số đo)
apriori(df_encoded, min_support=0.5, use_colnames=True)
fpgrowth(df_encoded, min_support=0.5, use_colnames=True)


def run_once(algo, minsup, minconf):
    """Chạy 1 lần: mining + sinh luật. Trả về (fi, rules, t_mining, t_total)."""
    t0 = time.time()
    fi = algo(df_encoded, min_support=minsup, use_colnames=True)
    t_mining = time.time() - t0
    rules = association_rules(fi, metric='confidence', min_threshold=minconf)
    t_total = time.time() - t0            # cách đo giống Apriori: mining + rules
    return fi, rules, t_mining, t_total


def rule_keys(rules):
    """Tập định danh luật (antecedents -> consequents) để so sánh chính xác."""
    return set(zip(rules['antecedents'], rules['consequents']))


def itemset_keys(fi):
    return set(fi['itemsets'])


# ---------- 3. CHẠY 9 CẤU HÌNH x 2 THUẬT TOÁN ----------
rows_ap, rows_fp, rows_cmp = [], [], []

for minsup in minsup_list:
    for minconf in minconf_list:
        res = {}
        for name, algo in [('Apriori', apriori), ('FP-Growth', fpgrowth)]:
            t_tot, t_min = [], []
            for _ in range(N_REPEAT):
                fi, rules, tm, tt = run_once(algo, minsup, minconf)
                t_min.append(tm)
                t_tot.append(tt)
            res[name] = dict(fi=fi, rules=rules,
                             t_total=float(np.median(t_tot)),
                             t_mining=float(np.median(t_min)))

        ap, fp = res['Apriori'], res['FP-Growth']

        # Bảng cùng format với Apriori
        rows_ap.append({'Minsup': minsup, 'Minconf': minconf,
                        'Số tập phổ biến': len(ap['fi']),
                        'Tổng số luật': len(ap['rules']),
                        'Thời gian (giây)': round(ap['t_total'], 4)})
        rows_fp.append({'Minsup': minsup, 'Minconf': minconf,
                        'Số tập phổ biến': len(fp['fi']),
                        'Tổng số luật': len(fp['rules']),
                        'Thời gian (giây)': round(fp['t_total'], 4)})

        # So sánh trực tiếp
        same_fi = itemset_keys(ap['fi']) == itemset_keys(fp['fi'])
        same_rules = rule_keys(ap['rules']) == rule_keys(fp['rules'])
        speedup = ap['t_total'] / fp['t_total'] if fp['t_total'] > 0 else np.nan
        rows_cmp.append({
            'Minsup': minsup, 'Minconf': minconf,
            'Apriori - Số tập PB': len(ap['fi']),
            'FP-Growth - Số tập PB': len(fp['fi']),
            'Apriori - Số luật': len(ap['rules']),
            'FP-Growth - Số luật': len(fp['rules']),
            'Cùng tập phổ biến?': 'Có' if same_fi else 'KHÔNG',
            'Cùng bộ luật?': 'Có' if same_rules else 'KHÔNG',
            'Apriori - TG mining (s)': round(ap['t_mining'], 4),
            'FP-Growth - TG mining (s)': round(fp['t_mining'], 4),
            'Apriori - TG tổng (s)': round(ap['t_total'], 4),
            'FP-Growth - TG tổng (s)': round(fp['t_total'], 4),
            'Speedup (Apriori/FP)': round(speedup, 2),
            'Nhanh hơn': 'FP-Growth' if fp['t_total'] < ap['t_total'] else 'Apriori',
        })
        print(f'minsup={minsup} minconf={minconf} | '
              f'FI {len(ap["fi"])}/{len(fp["fi"])} | '
              f'Rules {len(ap["rules"])}/{len(fp["rules"])} | '
              f'same_fi={same_fi} same_rules={same_rules} | '
              f'{ap["t_total"]:.3f}s vs {fp["t_total"]:.3f}s')

df_apriori = pd.DataFrame(rows_ap)
df_fpgrowth = pd.DataFrame(rows_fp)
df_compare = pd.DataFrame(rows_cmp)

print('\n=== BẢNG APRIORI ===')
print(df_apriori.to_string(index=False))
print('\n=== BẢNG FP-GROWTH (cùng format) ===')
print(df_fpgrowth.to_string(index=False))
print('\n=== SO SÁNH TRỰC TIẾP ===')
print(df_compare.to_string(index=False))

# ---------- 4. KẾT LUẬN TỰ ĐỘNG ----------
all_same = (df_compare['Cùng tập phổ biến?'].eq('Có').all()
            and df_compare['Cùng bộ luật?'].eq('Có').all())
print('\n=== KẾT LUẬN ===')
print('- Số tập phổ biến & số luật giống nhau ở cả 9 cấu hình:',
      'CÓ' if all_same else 'KHÔNG (cần kiểm tra lại)')
n_fp = (df_compare['Nhanh hơn'] == 'FP-Growth').sum()
print(f'- FP-Growth nhanh hơn ở {n_fp}/9 cấu hình; '
      f'speedup trung bình = {df_compare["Speedup (Apriori/FP)"].mean():.2f}x '
      f'(min {df_compare["Speedup (Apriori/FP)"].min():.2f}x, '
      f'max {df_compare["Speedup (Apriori/FP)"].max():.2f}x)')

# ---------- 5. LUẬT MỤC TIÊU CLASS (minsup=0.3, minconf=0.7) — kiểm tra khớp ----------
fi_fp = fpgrowth(df_encoded, min_support=0.3, use_colnames=True)
rules_fp = association_rules(fi_fp, metric='confidence', min_threshold=0.7)
mask = rules_fp['consequents'].apply(lambda s: any('class' in str(i).lower() for i in s))
rules_fp_target = rules_fp[mask]
print(f'\n-> FP-Growth (0.3/0.7): {len(rules_fp_target)} luật có consequents chứa class')

# ---------- 6. XUẤT EXCEL ĐỂ TỔNG HỢP ----------
out = 'FPGrowth_vs_Apriori_Results.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as w:
    df_apriori.to_excel(w, sheet_name='1. Apriori', index=False)
    df_fpgrowth.to_excel(w, sheet_name='2. FP-Growth', index=False)
    df_compare.to_excel(w, sheet_name='3. So sánh trực tiếp', index=False)
    # frozenset không ghi được vào Excel -> chuyển thành chuỗi
    t = rules_fp_target.copy()
    for c in ['antecedents', 'consequents']:
        t[c] = t[c].apply(lambda s: ', '.join(sorted(s)))
    t.to_excel(w, sheet_name='4. Luật class (FP-Growth)', index=False)
print(f"\n=> Đã xuất: '{out}'")
# ---------- 7. XUẤT BẢNG SO SÁNH RA CSV ----------
csv_out = 'FPGrowth_vs_Apriori_Comparison.csv'
df_compare.to_csv(csv_out, index=False, encoding='utf-8-sig')

print(f"=> Đã xuất bảng so sánh CSV: '{csv_out}'")


-> Dữ liệu: 8124 dòng, 22 cột
-> Ma trận one-hot: (8124, 118)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

minsup=0.3 minconf=0.6 | FI 1367/1367 | Rules 22602/22602 | same_fi=True same_rules=True | 0.269s vs 40.050s


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

minsup=0.3 minconf=0.7 | FI 1367/1367 | Rules 18876/18876 | same_fi=True same_rules=True | 0.269s vs 39.977s


...

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


-> FP-Growth (0.3/0.7): 3083 luật có consequents chứa class


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



=> Đã xuất: 'FPGrowth_vs_Apriori_Results.xlsx'
=> Đã xuất bảng so sánh CSV: 'FPGrowth_vs_Apriori_Comparison.csv'


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag